### Tutorial BuilDyn

Now we come to the main part, the BuilDyn class. This class combines most of the features mentioned before and serves as a API that can dynamically sample results from different vairations of FMUs with different walks and many more.

We start by initializing a BuilDyn. For that we need a FMU. Also, we already install the converter functions into the FMU object. To not do everything from scratch again, we use the function introduced in the notebook before.

In [ ]:
from builda_fmu import get_configured_builda_fmu

fmu = get_configured_builda_fmu()

In [ ]:
#
# Init BuilDyn
#
from buildyn.buildyn import BuilDyn

observables = ["thermalZone.TAir", "ctrSignalHeating"]

prior = BuilDyn(fmu, observables=observables)

Now that we initialized BuilDyn, we can use it so generate data from the FMU by only calling one function.

In [ ]:
df = prior.sample_one()

df["thermalZone.TAir"].plot()

Now this would be kind of boring. So we have other useful features to futher enhance the BuilDyn class.

#### Walker for BuilDyn

We can also initialize Walker from the notebook before into our BuilDyn. For instance:

In [ ]:
from buildyn.walker.fixed.poisson_walker import PoissonWalker
from buildyn.walker.interval_walker import IntervalWalker

random_walker = PoissonWalker(lam=8, is_discrete=False)
interval_walker = IntervalWalker(walker=random_walker, interval=900)

# For instance, we can add a walker for the heating control variable, just as in the FMU.
prior.add_walker_distribution("ctrSignalHeating", interval_walker)

When we sample from that now, we get a different curve every single time.

In [ ]:
df = prior.sample_one()
df["thermalZone.TAir"].plot()

**Distibution of Walkers**  

If we - for instance - want to use a different walker every single time we sample one from the BuilDyn, we can use a Distribution of Walkers like that:

In [ ]:
from buildyn.distributions.discrete.random_choice import RandomChoiceDistribution
from buildyn.walker.fixed.constant_walker import ConstantWalker

# Initialize a new BuilDyn
prior = BuilDyn(fmu=fmu, observables=observables)

# Define the walkers in the distribution.
p_walker = PoissonWalker(lam=8, is_discrete=False)
p_interval_walker = IntervalWalker(walker=p_walker, interval=900)

c_walker = ConstantWalker(constant=0.5)
c_interval_walker = IntervalWalker(walker=c_walker, interval=900)

# This can be any discrete distribution you want! For simplicity, we use one that just takes a walker a random.
dist = RandomChoiceDistribution([p_interval_walker, c_interval_walker])

prior.add_walker_distribution("ctrSignalHeating", dist)

If we sample multiple times from the prior again, we observe that the constant walker appears around half of the time, and the other half we see the Poisson Walker.

In [ ]:
df = prior.sample_one()
df["thermalZone.TAir"].plot()

#### Variations for BuilDyn

Oftentimes we not only want to change the distibution of walkers, but parameter of the system itself. In this case, we can Add distibutions over parameter in the BuilDyn as well. So each time we call the ```sample_one``` method, the FMU is parameterized in another way.

In [ ]:
from buildyn.distributions.continuous.gauss_distribution import GaussDistribution

g_dist = GaussDistribution(mu=1, sigma=0.5, min=0, max=3)

prior.add_variation_distribution("UExt", g_dist)

This code above chooses a value for the UExt of the FMU from a gauss distribution every simgle time a timeseries is sampled.

#### (RL) Random FMU Configurations.

Some use-cases might not want to use the timeseries generated but rather the (randomly) parameterized FMU from BuilDyn itself.  

In this cases, BuilDyn also provides a function.

In [ ]:
fmu1 = prior.sample_one_fmu()
fmu2 = prior.sample_one_fmu()

# We know that different UExt result in different heating capacities. So if the heatingCapacities change between the FMUs we know BuilDyn gave us different variations.
print(f"HeatingPower FMU1: {fmu1.get_variable('heatingPower')}")
print(f"HeatingPower FMU2: {fmu2.get_variable('heatingPower')}")

